<a href="https://colab.research.google.com/github/AndresAvella09/Proyecto_Multimodal/blob/main/notebooks/04_baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 04 — Experimento 1 (Tier A): unimodales vs fusión ingenua

**Pregunta:** ¿la fusión ingenua rinde igual que la mejor modalidad sola? Si sí, el modelo
multimodal no está aprovechando la segunda modalidad (*greedy modality*).

| Modelo | Entrada | Arquitectura |
|---|---|---|
| `imagen` | `clip_img` (512) | proyección 512→512 → ReLU → Dropout → 23 logits |
| `texto` | `distilbert_txt` (768) | proyección 768→512 → ReLU → Dropout → 23 logits |
| `fusion_ingenua` | ambos | [proy_img ; proy_txt] (1024) → ReLU → Dropout → 23 logits |

Las tres cabezas son la misma arquitectura (una proyección de 512 por modalidad + capa de salida),
así la única diferencia es qué modalidades ven.

**Protocolo** (para que la comparación sea justa y sin fuga de datos):

1. Estandarización con media y desviación calculadas **solo en train**.
2. Early stopping con el **AP macro de dev**: métrica sin umbral, no depende de cómo se corte.
3. **Umbrales por clase elegidos en dev** (maximizan F1 de cada género). Con un desbalance de
   42×, el umbral 0.5 hunde a las clases raras.
4. **Test se usa una sola vez**, al final, con los umbrales de dev.
5. 5 semillas → F1 macro en test como **media ± IC 95%** (t de Student, 4 g.l.).
6. La comparación clave (fusión − mejor unimodal) se hace **pareada**: por semilla y con
   bootstrap sobre las películas de test.

Corre en CPU en pocos minutos; no hace falta GPU.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, random, time
import numpy as np, pandas as pd, torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.metrics import average_precision_score

DATA = '/content/drive/MyDrive/proyecto_multimodal/data'
EMB = '/content/drive/MyDrive/proyecto_multimodal/embeddings'
RES = '/content/drive/MyDrive/proyecto_multimodal/results'
os.makedirs(RES, exist_ok=True)

df = pd.read_parquet(f'{DATA}/mmimdb_meta.parquet')
GENEROS = list(df.columns[4:])
Y = df[GENEROS].values.astype(np.float32)
X = {'img': np.load(f'{EMB}/clip_img.npy'), 'txt': np.load(f'{EMB}/distilbert_txt.npy')}
assert all(len(v) == len(df) for v in X.values())

idx = {s: np.where(df['particion'] == s)[0] for s in ['train', 'dev', 'test']}
print({s: len(i) for s, i in idx.items()}, "|", len(GENEROS), "géneros")
print({k: v.shape for k, v in X.items()})

plt.rcParams.update({'axes.spines.top': False, 'axes.spines.right': False,
                     'axes.grid': True, 'grid.alpha': 0.3, 'figure.dpi': 110})

## 1. Configuración

Todos los hiperparámetros en un solo lugar, iguales para los tres modelos. Más adelante pasan a
un YAML en `configs/`.

In [ ]:
CONFIG = {
    'dim_proy': 512,        # cada modalidad se proyecta a 512 (DistilBERT 768 -> 512)
    'dropout': 0.3,
    'lr': 1e-3,
    'weight_decay': 1e-4,
    'batch': 256,
    'max_epocas': 100,
    'paciencia': 10,        # épocas sin mejorar AP macro de dev antes de parar
    'semillas': [0, 1, 2, 3, 4],
    'umbrales': np.round(np.arange(0.05, 0.96, 0.05), 2),
    'n_bootstrap': 1000,
}
MODELOS = {'imagen': ['img'], 'texto': ['txt'], 'fusion_ingenua': ['img', 'txt']}
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print("Device:", device)

## 2. Estandarización (solo con train)

Las dos modalidades tienen escalas distintas (norma media ≈ 9 en CLIP vs ≈ 7.7 en DistilBERT, y
distinta varianza por dimensión). Sin estandarizar, la fusión podría favorecer a una rama solo
por escala numérica, y eso contaminaría la medición del desbalance.

In [ ]:
media = {m: X[m][idx['train']].mean(0) for m in X}
desv = {m: X[m][idx['train']].std(0) + 1e-6 for m in X}
Xz = {m: ((X[m] - media[m]) / desv[m]).astype(np.float32) for m in X}

def tensores(mods, s):
    return [torch.from_numpy(Xz[m][idx[s]]).to(device) for m in mods]

Yt = {s: torch.from_numpy(Y[idx[s]]).to(device) for s in idx}
for m in Xz:
    print(f"{m}: media train={Xz[m][idx['train']].mean():.3f}  desv train={Xz[m][idx['train']].std():.3f}  "
          f"| media test={Xz[m][idx['test']].mean():.3f}")

## 3. Modelo y métricas

In [ ]:
class Clasificador(nn.Module):
    # Una proyección lineal por modalidad -> concatenar -> ReLU -> Dropout -> 23 logits.
    # Con una sola modalidad es un MLP de 2 capas; con dos, la fusión ingenua (concatenación).
    def __init__(self, dims, dim_proy, dropout, n_clases=len(GENEROS)):
        super().__init__()
        self.proys = nn.ModuleList([nn.Linear(d, dim_proy) for d in dims])
        self.salida = nn.Sequential(nn.ReLU(), nn.Dropout(dropout),
                                    nn.Linear(dim_proy * len(dims), n_clases))

    def forward(self, xs):
        return self.salida(torch.cat([p(x) for p, x in zip(self.proys, xs)], dim=1))


def f1_por_clase(Y, P):
    # Y, P: (n, C) binarios -> (C,) F1 de cada clase (0 si la clase no aparece ni se predice)
    tp = (Y * P).sum(0); fp = ((1 - Y) * P).sum(0); fn = (Y * (1 - P)).sum(0)
    den = 2 * tp + fp + fn
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

def f1_micro(Y, P):
    tp = (Y * P).sum(); return 2 * tp / (2 * tp + ((1 - Y) * P).sum() + (Y * (1 - P)).sum())

def elegir_umbrales(Y, prob, grilla):
    # Para cada clase, el umbral de la grilla que maximiza su F1 en estos datos (dev)
    P = (prob[:, :, None] >= grilla[None, None, :]).astype(np.float32)   # (n, C, T)
    Yb = Y[:, :, None]
    tp = (Yb * P).sum(0); fp = ((1 - Yb) * P).sum(0); fn = (Yb * (1 - P)).sum(0)
    f1 = 2 * tp / np.maximum(2 * tp + fp + fn, 1)                         # (C, T)
    return grilla[f1.argmax(1)]

def metricas(Y, prob, umbrales):
    P = (prob >= umbrales).astype(np.float32)
    return {'f1_macro': f1_por_clase(Y, P).mean(), 'f1_micro': f1_micro(Y, P),
            'ap_macro': average_precision_score(Y, prob, average='macro')}

In [ ]:
@torch.no_grad()
def predecir(modelo, xs):
    modelo.eval()
    return torch.sigmoid(modelo(xs)).cpu().numpy()

def entrenar(mods, semilla, cfg=CONFIG):
    random.seed(semilla); np.random.seed(semilla); torch.manual_seed(semilla)
    xs_tr, xs_dev = tensores(mods, 'train'), tensores(mods, 'dev')
    modelo = Clasificador([X[m].shape[1] for m in mods], cfg['dim_proy'], cfg['dropout']).to(device)
    opt = torch.optim.AdamW(modelo.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    gen = torch.Generator(device='cpu').manual_seed(semilla)   # orden de los lotes reproducible

    n = len(idx['train']); mejor_ap, mejor_estado, sin_mejora = -1, None, 0
    hist = {'perdida_train': [], 'ap_dev': []}
    for epoca in range(cfg['max_epocas']):
        modelo.train(); perm = torch.randperm(n, generator=gen).to(device); total = 0.0
        for a in range(0, n, cfg['batch']):
            b = perm[a:a + cfg['batch']]
            perdida = F.binary_cross_entropy_with_logits(modelo([x[b] for x in xs_tr]), Yt['train'][b])
            opt.zero_grad(); perdida.backward(); opt.step()
            total += perdida.item() * len(b)
        ap = average_precision_score(Y[idx['dev']], predecir(modelo, xs_dev), average='macro')
        hist['perdida_train'].append(total / n); hist['ap_dev'].append(ap)
        if ap > mejor_ap:
            mejor_ap, sin_mejora = ap, 0
            mejor_estado = {k: v.detach().clone() for k, v in modelo.state_dict().items()}
        else:
            sin_mejora += 1
            if sin_mejora >= cfg['paciencia']:
                break
    modelo.load_state_dict(mejor_estado)
    hist['mejor_epoca'] = int(np.argmax(hist['ap_dev'])) + 1
    return modelo, hist

## 4. Entrenamiento: 3 modelos × 5 semillas

Por cada corrida: entrenar con early stopping en dev → elegir umbrales en dev → evaluar en test.
Se guardan las probabilidades de test de cada corrida: el diagnóstico (ECE por modalidad,
calibración) las va a reutilizar sin reentrenar.

In [ ]:
filas, probs_test, historias, umbrales_usados = [], {}, {}, {}
t0 = time.time()
for semilla in CONFIG['semillas']:
    for nombre, mods in MODELOS.items():
        modelo, hist = entrenar(mods, semilla)
        p_dev = predecir(modelo, tensores(mods, 'dev'))
        p_test = predecir(modelo, tensores(mods, 'test'))
        umb = elegir_umbrales(Y[idx['dev']], p_dev, CONFIG['umbrales'])
        m_dev = metricas(Y[idx['dev']], p_dev, umb)
        m_test = metricas(Y[idx['test']], p_test, umb)
        filas.append({'modelo': nombre, 'semilla': semilla, 'mejor_epoca': hist['mejor_epoca'],
                      **{f'dev_{k}': v for k, v in m_dev.items()},
                      **{f'test_{k}': v for k, v in m_test.items()}})
        probs_test[(nombre, semilla)] = p_test
        historias[(nombre, semilla)] = hist
        umbrales_usados[(nombre, semilla)] = umb
        print(f"semilla {semilla} | {nombre:15s} | época {hist['mejor_epoca']:3d} | "
              f"dev F1 macro {m_dev['f1_macro']:.4f} | test F1 macro {m_test['f1_macro']:.4f}")
print(f"\nTiempo total: {time.time() - t0:.0f} s")
res = pd.DataFrame(filas)

## 5. Resultados: media ± IC 95% sobre semillas

El IC refleja la variabilidad **del entrenamiento** (inicialización, orden de lotes). La
variabilidad por **muestra de test** se cubre con el bootstrap de la sección 6.

In [ ]:
def ic95(x):
    x = np.asarray(x, dtype=float); n = len(x)
    medio = stats.t.ppf(0.975, n - 1) * x.std(ddof=1) / np.sqrt(n)
    return x.mean(), medio

tabla = []
for nombre in MODELOS:
    r = res[res['modelo'] == nombre]; fila = {'modelo': nombre}
    for k in ['test_f1_macro', 'test_f1_micro', 'test_ap_macro', 'dev_f1_macro']:
        m, h = ic95(r[k]); fila[k] = f"{100 * m:.2f} ± {100 * h:.2f}"
    fila['épocas'] = f"{r['mejor_epoca'].mean():.0f}"
    tabla.append(fila)
tabla = pd.DataFrame(tabla).set_index('modelo')
tabla

## 6. La comparación clave: fusión − mejor unimodal

**Cuál es la "mejor unimodal" se decide con dev**, no con test: elegirla mirando test sería
usar test para seleccionar un modelo (fuga).

Dos IC complementarios para la diferencia de F1 macro:
- **Por semilla (pareado):** la diferencia en cada semilla, IC t sobre las 5 diferencias.
- **Bootstrap pareado sobre películas de test:** se re-muestrean películas con reemplazo, en cada
  re-muestreo se recalcula la diferencia de F1 macro para cada semilla y se promedia. Captura
  la incertidumbre por tener un test finito.

**Lectura:**
- Diferencia ≈ 0 (IC incluye 0) o muy chica → la fusión no aprovecha la segunda modalidad:
  evidencia de *greedy modality*.
- Diferencia claramente > 0 → la fusión sí suma; el desbalance se estudia induciéndolo con el
  dial de degradación (Exp. 4), como prevé la propuesta.

In [ ]:
media_dev = res.groupby('modelo')['dev_f1_macro'].mean()
mejor_uni = media_dev[['imagen', 'texto']].idxmax()
print(f"Mejor unimodal según dev: {mejor_uni} "
      f"(imagen {media_dev['imagen']:.4f} vs texto {media_dev['texto']:.4f})\n")

# IC por semilla, pareado
f = res[res['modelo'] == 'fusion_ingenua'].set_index('semilla')['test_f1_macro']
u = res[res['modelo'] == mejor_uni].set_index('semilla')['test_f1_macro']
m, h = ic95(f - u)
print(f"Fusión − {mejor_uni} (por semilla):  {100 * m:+.2f} ± {100 * h:.2f} puntos de F1 macro")

# Bootstrap pareado sobre películas de test
Yte = Y[idx['test']]
pred = {(n, s): (probs_test[(n, s)] >= umbrales_usados[(n, s)]).astype(np.float32)
        for n in ['fusion_ingenua', mejor_uni] for s in CONFIG['semillas']}
rng = np.random.default_rng(0)
difs = np.empty(CONFIG['n_bootstrap'])
for b in range(CONFIG['n_bootstrap']):
    r = rng.integers(0, len(Yte), len(Yte))
    difs[b] = np.mean([f1_por_clase(Yte[r], pred[('fusion_ingenua', s)][r]).mean()
                       - f1_por_clase(Yte[r], pred[(mejor_uni, s)][r]).mean()
                       for s in CONFIG['semillas']])
lo, hi = np.percentile(difs, [2.5, 97.5])
print(f"Fusión − {mejor_uni} (bootstrap test): {100 * difs.mean():+.2f}  IC 95% [{100 * lo:+.2f}, {100 * hi:+.2f}]")

## 7. ¿Dónde aporta cada modalidad? F1 por género

Promedio sobre semillas. Si la fusión sigue a la rama dominante incluso en los géneros donde la
otra rama es mejor, es una señal (todavía cualitativa) de supresión de la modalidad débil.

In [ ]:
f1_genero = {n: np.mean([f1_por_clase(Yte, pred_s).astype(float)
                         for pred_s in [(probs_test[(n, s)] >= umbrales_usados[(n, s)]).astype(np.float32)
                                        for s in CONFIG['semillas']]], axis=0)
             for n in MODELOS}
fg = pd.DataFrame(f1_genero, index=GENEROS)
fg['img_menos_txt'] = fg['imagen'] - fg['texto']
fg = fg.sort_values('img_menos_txt')

fig, ax = plt.subplots(figsize=(9, 8))
y = np.arange(len(fg))
for desp, n, c in [(-0.27, 'imagen', '#f59e0b'), (0, 'texto', '#2563eb'), (0.27, 'fusion_ingenua', '#10b981')]:
    ax.barh(y + desp, fg[n], height=0.27, label=n, color=c)
ax.set_yticks(y, fg.index, fontsize=8); ax.set_xlabel('F1 en test (media de 5 semillas)')
ax.set_title('F1 por género (ordenado por ventaja de imagen sobre texto)')
ax.grid(axis='y', visible=False); ax.legend(loc='lower right')
plt.tight_layout(); plt.show()

print("Géneros donde imagen supera a texto:")
print((fg[fg['img_menos_txt'] > 0][['imagen', 'texto', 'fusion_ingenua']] * 100).round(1).to_string())

## 8. Curvas de entrenamiento

AP macro de dev por época (semilla 0). Primera mirada a las *curvas de convergencia por
modalidad* que pide el diagnóstico: ¿qué rama aprende más rápido?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
for n, c in [('imagen', '#f59e0b'), ('texto', '#2563eb'), ('fusion_ingenua', '#10b981')]:
    h = historias[(n, 0)]
    axes[0].plot(range(1, len(h['perdida_train']) + 1), h['perdida_train'], color=c, label=n)
    axes[1].plot(range(1, len(h['ap_dev']) + 1), h['ap_dev'], color=c, label=n)
    axes[1].axvline(h['mejor_epoca'], color=c, ls=':', lw=1)
axes[0].set_title('Pérdida de entrenamiento (BCE)'); axes[0].set_xlabel('época')
axes[1].set_title('AP macro en dev (línea punteada = época elegida)'); axes[1].set_xlabel('época')
axes[1].legend(); plt.tight_layout(); plt.show()

## 9. Guardar resultados

- `exp1_baselines_por_semilla.csv`: métricas de cada corrida.
- `exp1_f1_por_genero.csv`: F1 por género promediado.
- `exp1_probs_test.npz`: probabilidades de test y umbrales de cada corrida (para calibración/ECE).

In [ ]:
res.to_csv(f'{RES}/exp1_baselines_por_semilla.csv', index=False)
fg.to_csv(f'{RES}/exp1_f1_por_genero.csv')
np.savez_compressed(f'{RES}/exp1_probs_test.npz',
                    **{f'prob_{n}_s{s}': p for (n, s), p in probs_test.items()},
                    **{f'umbral_{n}_s{s}': u for (n, s), u in umbrales_usados.items()},
                    idx_test=idx['test'], generos=np.array(GENEROS))
with open(f'{RES}/exp1_config.json', 'w') as fh:
    json.dump({k: (v.tolist() if isinstance(v, np.ndarray) else v) for k, v in CONFIG.items()}, fh, indent=2)
!ls -lh {RES}